# Squawk-Aligned Agile Modelling — Perch, Yucatan (self-contained)

Reproduces the Yucatan · Perch row of the dissertation's full-factorial design.
**No external `agile_protocol.py` import** — every step of the active-learning
loop is defined and called explicitly in this notebook.


In [ ]:
# =============================================================================
# Imports — standard scientific stack + perch_hoplite for the embedding DB
# and the linear classifier. No project-specific module is imported here;
# every helper used below is defined in this notebook.
# =============================================================================
import os, sys, json, sqlite3, re
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from perch_hoplite.agile import (
    audio_loader, classifier, classifier_data, embedding_display, source_info,
)
from perch_hoplite.db import (
    brutalism, interface, score_functions, search_results, sqlite_usearch_impl,
)
from perch_hoplite.zoo import model_configs

import importlib.metadata
print('perch-hoplite:', importlib.metadata.version('perch-hoplite'))


In [ ]:
# =============================================================================
# Paths + species config (Yucatan)
# =============================================================================
DB_PATH                 = 'C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings'
TRAIN_ANNOTATIONS_CSV   = 'C:/Users/Administrator/hello/Yucatan/yucatan_train_annotations.csv'
HOLDOUT_ANNOTATIONS_CSV = 'C:/Users/Administrator/hello/Yucatan/yucatan_test_annotations.csv'
SPECIES_CONFIG_PATH     = 'C:/Users/Administrator/hello/Yucatan/yucatan_species_config_stratified.json'
OUTPUT_ROOT             = 'C:/Users/Administrator/hello/Yucatan/Agile_results_squawk2_perch_explicit'

with open(SPECIES_CONFIG_PATH) as f:
    SPECIES_CONFIG = json.load(f)

# Yucatan recordings are time-expanded for Perch (10×), real-time for BD2.
TIME_SCALE  = 10.0
TOLERANCE_S = 0.25

print(f'Database:    {DB_PATH}')
print(f'Output root: {OUTPUT_ROOT}')
print(f'Time scale:  {TIME_SCALE}')
print(f'\nAvailable species:')
for sp_key, sp in SPECIES_CONFIG['species'].items():
    print(f"  {sp_key:<10} {sp.get('scientific_name', sp_key):<35} "
          f"({sp.get('n_training_annotations', '?')} train annotations)")


In [ ]:
# --- Inner classifier-training hyperparameters (paper §2.2) ---
# A single dense layer trained with binary cross-entropy loss, learning rate
# 1e-3, batch size 128, 128 steps, weak-negative weight 0.05, 90/10 train/val.
LEARNING_RATE   = 1e-3
NUM_STEPS       = 128
BATCH_SIZE      = 128
WEAK_NEG_WEIGHT = 0.05
LOSS_FN         = 'bce'
TRAIN_RATIO     = 0.9

# =============================================================================
# AGILE WORKFLOW HELPERS (paper §2.2)
#
# These are the pieces of the active-learning loop laid out as small named
# functions, so that the run loop below reads as one round of:
#   query → cosine search → auto-label → train classifier → score → select next
#
# Everything here is inlined directly into the notebook. No external module
# is imported.
# =============================================================================
import re
import sqlite3
from collections import defaultdict
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from perch_hoplite.agile import classifier, classifier_data, embedding_display
from perch_hoplite.db import brutalism, interface, score_functions, search_results


# --- filename / annotation helpers ---------------------------------------------

_CLIP_SUFFIX_RE = re.compile(r'_(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)\.wav$', re.IGNORECASE)


def split_clip_suffix(filename: str) -> Tuple[str, float]:
    """Recordings in the working folder are sometimes clipped: `foo_1.5_2.0.wav`
    means "the 0.5 s window of `foo.wav` starting at 1.5 s". Return the base
    filename and the clip start (0 if it's not a clip)."""
    fn = os.path.basename(str(filename))
    m = _CLIP_SUFFIX_RE.search(fn)
    if not m:
        return fn, 0.0
    return fn[:m.start()] + '.wav', float(m.group(1))


def build_annotation_index(csv_path: str, time_scale: float = 1.0) -> Dict[str, list]:
    """Read the annotation CSV and return a lookup of
    `{filename: [(start_s, end_s, label), ...]}`.

    `time_scale` converts annotation timestamps (real time) into the embedding
    time domain. Perch operates on 10× expanded audio so its timestamps are
    `time_scale=10`; BD2 operates on real-time audio so `time_scale=1`. Each
    clipped recording also gets an index entry under its base filename with
    timestamps shifted by the clip start, so we can match both naming styles.
    """
    df = pd.read_csv(csv_path).rename(columns={
        'recording_id': 'filename',
        'start_time': 'start_time_s',
        'end_time': 'end_time_s',
        'class': 'label',
    }).dropna(subset=['filename', 'start_time_s', 'end_time_s'])

    index = defaultdict(list)
    for row in df.itertuples(index=False):
        full_fn = os.path.basename(str(row.filename))
        base_fn, clip_start = split_clip_suffix(full_fn)
        s = float(row.start_time_s) * time_scale
        e = float(row.end_time_s) * time_scale
        label = str(row.label)
        index[full_fn].append((s, e, label))
        index[base_fn].append((s + clip_start, e + clip_start, label))

    for key in index:
        index[key].sort(key=lambda x: x[0])
    return dict(index)


def get_holdout_files(csv_path: str) -> Set[str]:
    """Return the set of base filenames in the held-out test split."""
    df = pd.read_csv(csv_path)
    fn_col = next(
        c for c in ('recording_id', 'filename', 'file', 'path', 'uri', 'source_id')
        if c in df.columns
    )
    return {split_clip_suffix(os.path.basename(str(fn)))[0]
            for fn in df[fn_col].astype(str)}


def label_for_window(
    filename: str,
    window_start_s: float,
    window_end_s: float,
    ann_index: Dict[str, list],
    positive_labels: Set[str],
    tolerance_s: float,
) -> Optional[bool]:
    """Auto-label a window from the annotation timestamps.

    Returns `True` if any annotation that overlaps the window (with ±tolerance)
    is a positive label, `False` if the window overlaps annotations but none
    are positive, and `None` if the file has no annotations at all (no GT
    coverage). The paper uses `tolerance_s = 0.025` (real time) and treats files
    without coverage as negatives further upstream — see `auto_label_candidates`.
    """
    fn = os.path.basename(str(filename))
    base_fn, clip_start = split_clip_suffix(fn)

    events = ann_index.get(fn)
    if events:
        hit = [lbl for (s, e, lbl) in events
               if (e + tolerance_s) >= window_start_s
               and (s - tolerance_s) <= window_end_s]
        if not hit:
            return False
        return any(lbl in positive_labels for lbl in hit)

    events = ann_index.get(base_fn)
    if events is None:
        return None

    shifted_start = window_start_s + clip_start
    shifted_end = window_end_s + clip_start
    hit = [lbl for (s, e, lbl) in events
           if (e + tolerance_s) >= shifted_start
           and (s - tolerance_s) <= shifted_end]
    if not hit:
        return False
    return any(lbl in positive_labels for lbl in hit)


# --- label management ----------------------------------------------------------

def get_labelled_embedding_ids(db, label_text: str) -> Set[int]:
    """Return embedding IDs that already have a label for `label_text`."""
    rows = db.db.execute(
        'SELECT embedding_id FROM hoplite_labels WHERE label = ?',
        (label_text,),
    ).fetchall()
    return {int(r[0]) for r in rows}


def count_labels_by_type(db, label_text: str) -> Dict[str, int]:
    """Count positive vs negative labels currently stored for `label_text`."""
    counts = {'positive': 0, 'negative': 0}
    for label_type, n in db.db.execute(
        'SELECT type, COUNT(*) FROM hoplite_labels WHERE label = ? GROUP BY type',
        (label_text,),
    ).fetchall():
        if label_type == interface.LabelType.POSITIVE.value:
            counts['positive'] = n
        elif label_type == interface.LabelType.NEGATIVE.value:
            counts['negative'] = n
    return counts


def delete_labels_by_text(db, label_text: str) -> int:
    """Wipe every label for `label_text`. Called between seeds so the per-seed
    classifier starts from a clean slate."""
    n = db.db.execute(
        'SELECT COUNT(*) FROM hoplite_labels WHERE label = ?',
        (label_text,),
    ).fetchone()[0]
    db.db.execute('DELETE FROM hoplite_labels WHERE label = ?', (label_text,))
    db.db.commit()
    return n


def insert_label(db, embedding_id: int, label_text: str, is_positive: bool, provenance: str) -> bool:
    """Insert one (embedding, label) pair. Returns True on insert,
    False if a duplicate already exists."""
    label_type = interface.LabelType.POSITIVE if is_positive else interface.LabelType.NEGATIVE
    return db.insert_label(
        interface.Label(
            embedding_id=int(embedding_id),
            label=str(label_text),
            type=label_type,
            provenance=str(provenance),
        ),
        skip_duplicates=True,
    )


# --- scoring + active-learning selection (paper §2.4.1) ------------------------

def score_all_embeddings(db, linear_classifier, target_label: str,
                         data_manager) -> Dict[int, float]:
    """Compute the classifier logit for every embedding in the database in
    batches. Returns `{embedding_id: logit}`."""
    target_idx = data_manager.get_target_labels().index(target_label)
    weights = np.asarray(linear_classifier.beta[:, target_idx])
    bias = float(linear_classifier.beta_bias[target_idx])

    scores: Dict[int, float] = {}
    all_ids = db.get_embedding_ids()
    for start in range(0, len(all_ids), 2048):
        batch_ids = np.asarray(all_ids[start:start + 2048])
        ids_arr, embs = db.get_embeddings(batch_ids)
        logits = embs @ weights + bias
        for eid, score in zip(ids_arr, logits):
            scores[int(eid)] = float(score)
    return scores


def select_top50(scores: Dict[int, float], n: int, exclude_ids: Set[int]) -> List[int]:
    """Take the n highest-scoring unlabelled embeddings."""
    candidates = sorted(
        ((eid, s) for eid, s in scores.items() if eid not in exclude_ids),
        key=lambda x: x[1],
        reverse=True,
    )
    return [eid for eid, _ in candidates[:n]]


def select_top10_quantile(scores: Dict[int, float], n: int, exclude_ids: Set[int],
                          seed: int, n_top: int = 10) -> List[int]:
    """Top-n_top by score, then distribute (n − n_top) across six logarithmic
    quantile buckets (50%, 25%, 12.5%, 6.25%, 3.125%, 1.5625%) of the remaining
    score distribution, sampling randomly within each bucket."""
    rng = np.random.default_rng(seed)
    ranked = sorted(
        ((eid, s) for eid, s in scores.items() if eid not in exclude_ids),
        key=lambda x: x[1],
        reverse=True,
    )
    if len(ranked) <= n:
        return [eid for eid, _ in ranked]

    selected = [eid for eid, _ in ranked[:n_top]]
    remaining = ranked[n_top:]
    n_buckets = 6
    fracs = np.array([0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625])
    fracs = fracs / fracs.sum()

    buckets, cum = [], 0.0
    for frac in fracs:
        lo = int(cum * len(remaining))
        cum += frac
        hi = min(int(cum * len(remaining)), len(remaining))
        buckets.append(remaining[lo:hi])

    quota = n - n_top
    base = quota // n_buckets
    extra = quota % n_buckets
    per_bucket = [base + (1 if i < extra else 0) for i in range(n_buckets)]

    for bucket, k in zip(buckets, per_bucket):
        if not bucket or k == 0:
            continue
        idx = rng.choice(len(bucket), size=min(k, len(bucket)), replace=False)
        for j in idx:
            selected.append(bucket[j][0])

    return selected[:n]


def select_balanced_boundary(
    scores: Dict[int, float],
    exclude_ids: Set[int],
    top_k: int = 50,
    boundary_k: int = 150,
    num_review: int = 200,
) -> List[int]:
    """Review pool for the balanced-boundary strategy: the top top_k by score
    plus the boundary_k closest to the decision boundary (|score| smallest),
    de-duplicated, truncated to num_review. The downstream label inserter caps
    to balanced_n_pos + balanced_n_neg = 20 labels per iteration."""
    candidates = [(eid, s) for eid, s in scores.items() if eid not in exclude_ids]
    if not candidates:
        return []

    top_ids = [eid for eid, _ in sorted(candidates, key=lambda x: x[1], reverse=True)[:top_k]]
    top_set = set(top_ids)
    boundary_ids = [
        eid for eid, _ in sorted(
            ((eid, s) for eid, s in candidates if eid not in top_set),
            key=lambda x: abs(x[1]),
        )[:boundary_k]
    ]
    return (top_ids + boundary_ids)[:num_review]


# --- auto-labelling (paper §2.4.3: optional 20% label noise) -------------------

def auto_label_candidates(
    db,
    embedding_ids: List[int],
    candidate_scores: Optional[Dict[int, float]],
    target_label: str,
    ann_index: Dict[str, list],
    positive_labels: Set[str],
    holdout_base_files: Set[str],
    window_size_s: float,
    tolerance_s: float,
    noise_flip_rate: float,
    seed: int,
    provenance: str,
    max_pos: Optional[int] = None,
    max_neg: Optional[int] = None,
) -> Dict[str, int]:
    """Look up the ground-truth label for each candidate by timestamp overlap
    and insert it into the database.

    - Embeddings whose source recording is in the holdout set are skipped.
    - Files with no annotations at all are treated as negatives ("file was
      reviewed, target species not found").
    - `noise_flip_rate` (paper §2.4.3) randomly flips that fraction of labels
      before insertion to simulate annotator error.
    - For the balanced-boundary strategy, `max_pos`/`max_neg` cap how many of
      each label class go in. Positives are ranked by highest score (most
      confident likely positives), negatives by |score| ascending (boundary-
      closest). `candidate_scores` must be supplied when caps are active.
    """
    rng = np.random.default_rng(seed)
    already_labelled = get_labelled_embedding_ids(db, target_label)

    rows = []
    for eid in embedding_ids:
        if eid in already_labelled:
            continue
        src = db.get_embedding_source(eid)
        if src is None:
            continue
        filename = os.path.basename(str(src.source_id))
        base_fn, clip_start = split_clip_suffix(filename)
        if base_fn in holdout_base_files:
            continue

        offset_s = float(src.offsets[0]) if src.offsets is not None else 0.0
        w0 = clip_start + offset_s
        w1 = w0 + window_size_s
        gt = label_for_window(filename, w0, w1, ann_index, positive_labels, tolerance_s)
        if gt is None:
            gt = False  # file not in annotation index → treat as negative

        score = float(candidate_scores.get(eid, 0.0)) if candidate_scores else 0.0
        rows.append({'embedding_id': eid, 'is_positive': bool(gt), 'score': score})

    if not rows:
        return {'n_inserted': 0, 'n_pos': 0, 'n_neg': 0, 'n_flipped': 0}

    # Optional label-noise injection (random flip)
    n_flipped = 0
    if noise_flip_rate > 0:
        n_to_flip = max(1, int(len(rows) * noise_flip_rate))
        for j in rng.choice(len(rows), size=min(n_to_flip, len(rows)), replace=False):
            rows[j]['is_positive'] = not rows[j]['is_positive']
            n_flipped += 1

    # balanced-boundary caps: positives by top score, negatives by smallest |score|.
    if max_pos is not None or max_neg is not None:
        pos = sorted((r for r in rows if r['is_positive']),
                     key=lambda r: r['score'], reverse=True)
        neg = sorted((r for r in rows if not r['is_positive']),
                     key=lambda r: abs(r['score']))
        if max_pos is not None:
            pos = pos[:max_pos]
        if max_neg is not None:
            neg = neg[:max_neg]
        rows = pos + neg

    n_inserted = 0
    for r in rows:
        if insert_label(db, r['embedding_id'], target_label, r['is_positive'], provenance):
            n_inserted += 1
    db.db.commit()

    n_pos = sum(1 for r in rows if r['is_positive'])
    n_neg = sum(1 for r in rows if not r['is_positive'])
    return {'n_inserted': n_inserted, 'n_pos': n_pos, 'n_neg': n_neg, 'n_flipped': n_flipped}


# --- classifier training + evaluation (paper §2.2, §2.5) -----------------------

def train_linear_classifier(
    db, target_label: str, *,
    learning_rate: float = 1e-3,
    num_steps: int = 128,
    batch_size: int = 128,
    weak_neg_weight: float = 0.05,
    loss: str = 'bce',
    train_ratio: float = 0.9,
    seed: int = 42,
):
    """Train the single-dense-layer classifier (paper §2.2). Returns
    `(linear_classifier, data_manager)`."""
    data_manager = classifier_data.AgileDataManager(
        target_labels=(target_label,),
        db=db,
        train_ratio=train_ratio,
        min_eval_examples=2,
        batch_size=batch_size,
        weak_negatives_batch_size=batch_size,
        rng=np.random.default_rng(seed),
    )
    linear_classifier, _ = classifier.train_linear_classifier(
        data_manager,
        learning_rate=learning_rate,
        weak_neg_weight=weak_neg_weight,
        num_train_steps=num_steps,
        loss=loss,
    )
    return linear_classifier, data_manager


def evaluate_on_holdout(
    scores: Dict[int, float],
    db,
    holdout_files: Set[str],
    holdout_ann_index: Dict[str, list],
    positive_labels: Set[str],
    window_size_s: float,
    tolerance_s: float,
) -> Dict[str, float]:
    """Score the held-out test windows and compute AP + ROC-AUC.
    Bootstrap confidence intervals are NOT computed here — they're produced
    downstream in `04_statistical_tests/` from the per-run AP values across
    species and seeds (paper §2.5)."""
    y_true: List[bool] = []
    y_score: List[float] = []

    for eid, score in scores.items():
        src = db.get_embedding_source(eid)
        if src is None:
            continue
        filename = os.path.basename(str(src.source_id))
        base_fn, clip_start = split_clip_suffix(filename)
        if base_fn not in holdout_files:
            continue

        offset_s = float(src.offsets[0]) if src.offsets is not None else 0.0
        w0 = clip_start + offset_s
        w1 = w0 + window_size_s
        gt = label_for_window(filename, w0, w1, holdout_ann_index, positive_labels, tolerance_s)
        if gt is None:
            continue
        y_true.append(bool(gt))
        y_score.append(score)

    n_pos = int(sum(y_true))
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return {'avg_precision': 0.0, 'roc_auc': 0.5,
                'n_test_windows': len(y_true),
                'n_test_positives': n_pos, 'n_test_negatives': n_neg}

    y_true_arr = np.asarray(y_true, dtype=bool)
    y_score_arr = np.asarray(y_score, dtype=float)
    return {
        'avg_precision': float(average_precision_score(y_true_arr, y_score_arr)),
        'roc_auc': float(roc_auc_score(y_true_arr, y_score_arr)),
        'n_test_windows': len(y_true),
        'n_test_positives': n_pos,
        'n_test_negatives': n_neg,
    }


In [ ]:
# =============================================================================
# PERCH RUNNER
# Connects to the Perch embedding database, builds the train/test annotation
# indices, and looks up query embeddings by source filename + offset.
# Query embeddings come straight from the database (not re-embedded from audio)
# because the DB was built at 32 kHz without resampling — re-embedding would
# resample from the native ultrasonic sample rate, producing a completely
# different vector.
# =============================================================================

class PerchAgileRunner:
    """State container for one Perch · dataset run. The active-learning loop
    above pulls shared state (db, classifier, annotation indices, query
    metadata) off this object."""

    def __init__(
        self,
        db_path: str,
        train_annotations_csv: str,
        holdout_annotations_csv: str,
        target_label: str,
        positive_labels: Set[str],
        time_scale: float,
        tolerance_s: float,
        output_root: str,
    ):
        self.db_path = db_path
        self.train_annotations_csv = train_annotations_csv
        self.holdout_annotations_csv = holdout_annotations_csv
        self.target_label = target_label
        self.positive_labels = set(positive_labels)
        self.time_scale = float(time_scale)
        self.tolerance_s = float(tolerance_s)
        self.output_root = Path(output_root)
        self.output_root.mkdir(parents=True, exist_ok=True)

        # Filled in by `setup()`
        self.db = None
        self.window_size_s = None
        self.sample_rate = None
        self.train_ann_index = None
        self.holdout_ann_index = None
        self.holdout_files = None

        # Per-run state
        self.query_source_id = None
        self.query_offset_s = 0.0
        self.linear_classifier = None
        self.data_manager = None
        self.run_id = None

    def setup(self):
        """Connect to the embedding DB and build the annotation indices."""
        self.db = sqlite_usearch_impl.SQLiteUsearchDB.create(self.db_path)
        db_model_config = self.db.get_metadata('model_config')
        model_class = model_configs.get_model_class(db_model_config.model_key)
        embedding_model = model_class.from_config(db_model_config.model_config)
        self.window_size_s = float(getattr(embedding_model, 'window_size_s', 5.0))
        self.sample_rate = int(embedding_model.sample_rate)

        self.train_ann_index = build_annotation_index(
            self.train_annotations_csv, time_scale=self.time_scale,
        )
        self.holdout_ann_index = build_annotation_index(
            self.holdout_annotations_csv, time_scale=self.time_scale,
        )
        self.holdout_files = get_holdout_files(self.holdout_annotations_csv)

        print(f'  DB: {self.db_path}  ({self.db.count_embeddings()} embeddings, '
              f'{self.db.embedding_dimension()}-d)')
        print(f'  Train annotations: {len(self.train_ann_index)} files')
        print(f'  Holdout files: {len(self.holdout_files)}')
        print(f'  Window {self.window_size_s} s · sample rate {self.sample_rate} Hz · '
              f'time_scale {self.time_scale}')

    def set_query(self, query_config: dict):
        """Switch the query to a different recording + offset, called between seeds."""
        self.query_source_id = str(os.path.basename(query_config['recording_id']))
        self.query_offset_s = float(query_config.get('offset_s', 0.0))

    def get_query_embedding(self) -> np.ndarray:
        """Look up the embedding closest to (query_source_id, query_offset_s)
        directly from the DB."""
        conn = sqlite3.connect(str(self.db._sqlite_filepath))
        try:
            rows = conn.execute(
                'SELECT e.id, e.offsets FROM hoplite_embeddings e '
                'JOIN hoplite_sources s ON e.source_idx = s.id '
                'WHERE s.source = ?',
                (self.query_source_id,),
            ).fetchall()
        finally:
            conn.close()
        if not rows:
            raise ValueError(f'Query source {self.query_source_id!r} not found in DB')

        best_id, best_dist = None, float('inf')
        for eid, offsets_blob in rows:
            offset = float(np.frombuffer(offsets_blob, dtype=np.float64)[0])
            dist = abs(offset - self.query_offset_s)
            if dist < best_dist:
                best_dist = dist
                best_id = int(eid)
        return self.db.get_embedding(best_id).astype(np.float32)


In [ ]:
# =============================================================================
# THE ACTIVE-LEARNING WORKFLOW (paper §2.2, Figure 1)
#
# One Agile run = 5 iterations on one (species, AL strategy, SNR stratum,
# label-noise condition, seed) cell. Each iteration is:
#
#   iter 0  : query embedding → cosine nearest-neighbour search → auto-label →
#             train linear classifier → score all embeddings →
#             evaluate AP/AUC on the held-out test set
#   iter 1+ : score all embeddings with the current classifier →
#             select next 50 (or up to 200 reviewed for balanced-boundary) →
#             auto-label → re-train → re-score → re-evaluate
#
# The classifier is rebuilt from the cumulative label set each iteration.
# Labels are wiped between seeds so every run starts from zero.
# =============================================================================
def run_one_iteration(
    runner,
    *,
    iter_idx: int,
    al_strategy: str,
    noise_flip_rate: float,
    seed: int,
    balanced_n_pos: int = 10,
    balanced_n_neg: int = 10,
    balanced_top_k: int = 50,
    balanced_boundary_k: int = 150,
    balanced_num_review: int = 200,
    n_select: int = 50,
) -> Dict[str, float]:
    """Run a single active-learning iteration end-to-end and return the
    test-set AP/AUC plus label counts. The runner holds the shared state
    (DB, classifier from previous iteration, annotation indices, etc.)."""

    excluded_ids = get_labelled_embedding_ids(runner.db, runner.target_label)

    # --- 1. Pick candidates to label ------------------------------------------
    if iter_idx == 0:
        # First iteration: there is no classifier yet, so seed with a
        # cosine-similarity search from the query embedding.
        query_emb = runner.get_query_embedding()
        results, _ = brutalism.threaded_brute_search(
            runner.db, query_emb, n_select,
            score_fn=score_functions.get_score_fn('cos'),
            sample_size=1_000_000,
        )
        candidate_ids = [int(m.embedding_id) for m in results.search_results]
        candidate_scores = {int(m.embedding_id): float(m.sort_score)
                             for m in results.search_results}
        max_pos, max_neg = None, None
    else:
        # Subsequent iterations: score the whole DB with the current classifier
        # and let the active-learning strategy choose what to label next.
        scores = score_all_embeddings(
            runner.db, runner.linear_classifier,
            runner.target_label, runner.data_manager,
        )
        if al_strategy == 'top50':
            candidate_ids = select_top50(scores, n_select, excluded_ids)
            max_pos, max_neg = None, None
        elif al_strategy == 'top10_quantile':
            candidate_ids = select_top10_quantile(scores, n_select, excluded_ids, seed=seed)
            max_pos, max_neg = None, None
        elif al_strategy == 'balanced_boundary':
            candidate_ids = select_balanced_boundary(
                scores, excluded_ids,
                top_k=balanced_top_k,
                boundary_k=balanced_boundary_k,
                num_review=balanced_num_review,
            )
            max_pos, max_neg = balanced_n_pos, balanced_n_neg
        else:
            raise ValueError(f'Unknown al_strategy: {al_strategy!r}')
        candidate_scores = scores

    # --- 2. Auto-label the candidates -----------------------------------------
    provenance = f'{runner.run_id}__iter{iter_idx:02d}__{al_strategy}'
    label_summary = auto_label_candidates(
        runner.db,
        candidate_ids,
        candidate_scores=candidate_scores,
        target_label=runner.target_label,
        ann_index=runner.train_ann_index,
        positive_labels=runner.positive_labels,
        holdout_base_files=runner.holdout_files,
        window_size_s=runner.window_size_s,
        tolerance_s=runner.tolerance_s,
        noise_flip_rate=noise_flip_rate,
        seed=seed,
        provenance=provenance,
        max_pos=max_pos,
        max_neg=max_neg,
    )

    # --- 3. Retrain the classifier on the cumulative labels --------------------
    counts = count_labels_by_type(runner.db, runner.target_label)
    if counts['positive'] == 0 or counts['negative'] == 0:
        # Need ≥1 of each class to train. This only happens when the seed
        # query retrieves zero positives, which is rare but possible.
        metrics = {'avg_precision': 0.0, 'roc_auc': 0.5,
                   'n_test_windows': 0, 'n_test_positives': 0, 'n_test_negatives': 0}
        runner.linear_classifier = None
        runner.data_manager = None
    else:
        runner.linear_classifier, runner.data_manager = train_linear_classifier(
            runner.db, runner.target_label,
            learning_rate=LEARNING_RATE,
            num_steps=NUM_STEPS,
            batch_size=BATCH_SIZE,
            weak_neg_weight=WEAK_NEG_WEIGHT,
            loss=LOSS_FN,
            train_ratio=TRAIN_RATIO,
            seed=seed,
        )
        # Score every embedding and pick out the held-out test windows.
        scores_for_eval = score_all_embeddings(
            runner.db, runner.linear_classifier,
            runner.target_label, runner.data_manager,
        )
        metrics = evaluate_on_holdout(
            scores_for_eval, runner.db,
            holdout_files=runner.holdout_files,
            holdout_ann_index=runner.holdout_ann_index,
            positive_labels=runner.positive_labels,
            window_size_s=runner.window_size_s,
            tolerance_s=runner.tolerance_s,
        )

    return {
        **metrics,
        'iter_idx': iter_idx,
        'n_labeled_this_iter': label_summary['n_inserted'],
        'pos_added': label_summary['n_pos'],
        'neg_added': label_summary['n_neg'],
        'flipped': label_summary['n_flipped'],
        'cumulative_pos': counts['positive'],
        'cumulative_neg': counts['negative'],
        'cumulative_total': counts['positive'] + counts['negative'],
    }


def run_one_agile_run(
    runner,
    *,
    n_iters: int = 5,
    al_strategy: str = 'balanced_boundary',
    noise_flip_rate: float = 0.0,
    seed: int = 0,
    query_config: Optional[dict] = None,
    n_select: int = 50,
    balanced_n_pos: int = 10,
    balanced_n_neg: int = 10,
    balanced_top_k: int = 50,
    balanced_boundary_k: int = 150,
    balanced_num_review: int = 200,
) -> pd.DataFrame:
    """One full active-learning run (5 iterations) on one cell. Wipes any
    pre-existing labels for this species before starting so the run is
    independent of previous seeds."""

    # Swap query if provided (caller does this between seeds).
    if query_config is not None:
        runner.set_query(query_config)

    delete_labels_by_text(runner.db, runner.target_label)
    runner.linear_classifier = None
    runner.data_manager = None
    runner.run_id = (f'{datetime.now():%Y%m%d_%H%M%S}_{runner.target_label}'
                     f'_{al_strategy}_seed{seed}')

    rows = []
    for iter_idx in range(n_iters):
        # Each iteration uses a different RNG seed for AL selection / label-flip.
        result = run_one_iteration(
            runner,
            iter_idx=iter_idx,
            al_strategy=al_strategy,
            noise_flip_rate=noise_flip_rate,
            seed=seed + iter_idx,
            balanced_n_pos=balanced_n_pos,
            balanced_n_neg=balanced_n_neg,
            balanced_top_k=balanced_top_k,
            balanced_boundary_k=balanced_boundary_k,
            balanced_num_review=balanced_num_review,
            n_select=n_select,
        )
        result.update({
            'run_id': runner.run_id,
            'al_strategy': al_strategy,
            'noise_flip_rate': noise_flip_rate,
            'seed': seed,
            'protocol_seed': seed,
            'timestamp': datetime.now().isoformat(),
        })
        if query_config is not None:
            result['query_recording_id'] = query_config.get('recording_id', '')
            result['query_wada_snr_db'] = query_config.get('wada_snr_db', np.nan)
        rows.append(result)

        print(f'  iter {iter_idx}: AP={result["avg_precision"]:.3f}  '
              f'AUC={result["roc_auc"]:.3f}  '
              f'labels=+{result["n_labeled_this_iter"]} '
              f'(cum {result["cumulative_total"]})')

    return pd.DataFrame(rows)


In [ ]:
# =============================================================================
# OUTER BATCH LOOP (paper §2.4 — full-factorial design)
#
# Iterates over the 18 conditions × 5 seeds = 90 runs per species and writes
# results into `OUTPUT_ROOT/<species>/<condition>/<seed>__run_summary.csv`.
# When the whole grid is filled, also writes a rollup `all_conditions_summary.csv`.
#
# Resume behaviour: any (condition, seed) that already has a run_summary.csv is
# skipped. Delete that CSV to force a re-run for that one cell.
# =============================================================================
AL_STRATEGIES = [
    {'al_strategy': 'top50'},
    {'al_strategy': 'top10_quantile'},
    {'al_strategy': 'balanced_boundary',
     'balanced_n_pos': 10, 'balanced_n_neg': 10,
     'balanced_top_k': 50, 'balanced_boundary_k': 150,
     'balanced_num_review': 200},
]

NOISE_LEVELS = [
    {'noise_flip_rate': 0.0, 'label': 'cleanL'},
    {'noise_flip_rate': 0.2, 'label': 'noisyL'},
]

STRATA = ['high', 'mid', 'low']


def run_full_factorial(runner, species_key: str, species_queries_by_stratum: dict,
                       output_root: str) -> pd.DataFrame:
    """Run all 18 × 5 = 90 cells for one species and return the concatenated
    per-iteration DataFrame. Atomically saves each cell so a kernel restart
    loses at most one cell's work."""
    species_dir = Path(output_root) / species_key
    species_dir.mkdir(parents=True, exist_ok=True)

    all_rows = []
    cond_idx = 0
    for stratum in STRATA:
        queries = species_queries_by_stratum[stratum]
        for al_cfg in AL_STRATEGIES:
            al_name = al_cfg['al_strategy']
            al_short = 'balanced' if al_name == 'balanced_boundary' else al_name
            balanced_kwargs = {k: v for k, v in al_cfg.items() if k != 'al_strategy'}
            for noise_cfg in NOISE_LEVELS:
                cond_idx += 1
                cond_name = (f'C{cond_idx}_{al_short}_{stratum}Q_{noise_cfg["label"]}')
                cond_dir = species_dir / cond_name
                cond_dir.mkdir(parents=True, exist_ok=True)

                print(f'\n=== condition {cond_idx}/18: {cond_name} ===')
                cond_rows = []
                for seed in range(5):
                    seed_csv = cond_dir / f'seed{seed}__run_summary.csv'
                    if seed_csv.exists():
                        print(f'  seed {seed}: skip (cached)')
                        cond_rows.append(pd.read_csv(seed_csv))
                        continue

                    print(f'  seed {seed}: query {queries[seed]["recording_id"]} '
                          f'@ {queries[seed]["offset_s"]}s  '
                          f'(SNR {queries[seed]["wada_snr_db"]:.1f} dB)')
                    df = run_one_agile_run(
                        runner,
                        n_iters=5,
                        al_strategy=al_name,
                        noise_flip_rate=noise_cfg['noise_flip_rate'],
                        seed=seed,
                        query_config=queries[seed],
                        **balanced_kwargs,
                    )
                    df['condition'] = cond_name
                    df['stratum'] = stratum
                    df['noise_label'] = noise_cfg['label']
                    df['species'] = species_key
                    df.to_csv(seed_csv, index=False)
                    cond_rows.append(df)

                cond_df = pd.concat(cond_rows, ignore_index=True)
                all_rows.append(cond_df)

                final_ap = cond_df.loc[cond_df.iter_idx == cond_df.iter_idx.max(),
                                       'avg_precision']
                print(f'  >> {cond_name}: mean final-iter AP = '
                      f'{final_ap.mean():.3f} ± {final_ap.std():.3f}  (n={len(final_ap)})')

    rollup = pd.concat(all_rows, ignore_index=True)
    rollup.to_csv(species_dir / 'all_conditions_summary.csv', index=False)
    print(f'\nSaved rollup: {species_dir / "all_conditions_summary.csv"}  ({len(rollup)} rows)')
    return rollup


In [ ]:
# =============================================================================
# Run the full 90-cell factorial for every Yucatan species.
# =============================================================================
SPECIES_LIST = ['ept_fur', 'eum_und', 'las_ega', 'mor_meg',
                'myo_kea', 'pte_mac', 'pte_par', 'sac_bil']

for SPECIES_KEY in SPECIES_LIST:
    species = SPECIES_CONFIG['species'][SPECIES_KEY]
    target_label = SPECIES_KEY
    positive_labels = set(species['positive_label_values'])
    queries_by_stratum = {s: species['strata'][s]['queries'] for s in ('high', 'mid', 'low')}

    print(f'\n{"#"*72}\n# SPECIES {SPECIES_KEY}\n{"#"*72}')

    runner = PerchAgileRunner(
        db_path=DB_PATH,
        train_annotations_csv=TRAIN_ANNOTATIONS_CSV,
        holdout_annotations_csv=HOLDOUT_ANNOTATIONS_CSV,
        target_label=target_label,
        positive_labels=positive_labels,
        time_scale=TIME_SCALE,
        tolerance_s=TOLERANCE_S,
        output_root=OUTPUT_ROOT,
    )
    runner.setup()
    run_full_factorial(runner, SPECIES_KEY, queries_by_stratum, OUTPUT_ROOT)
